In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless – swap to "TkAgg" for interactive use
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.analyses.quantization import LOManalysis, EPRanalysis

In [2]:
#Define a chip and the design size extends from the origin to the right quadrant.

In [3]:
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '2mm'
design.chips.main.size['size_y'] = '2mm'

gui = MetalGUI(design)

In [4]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

design.delete_all_components()

# Standard "Golden" values for a ~5GHz Transmon
q1 = TransmonPocket(design, 'Q1', options = {
    'pos_x': '0um',
    'pos_y': '0um',
    'pad_width': '425um',    # Standard width
    'pad_height': '90um',    # Standard height
    'pad_gap': '30um',       # Standard gap to ground
    'jj_width': '200nm',     # Realistic junction width
    'jj_height': '200nm',    # Realistic junction height
    'pocket_height': '650um',
    'connection_pads': {}
})

gui.rebuild()
gui.autoscale()

In [5]:
import numpy as np

def verify_qubit_physics(qubit_obj):
    # --- Physical Constants ---
    h = 6.626e-34; e = 1.602e-19; phi0 = 2.067833848e-15; eps0 = 8.854187e-12
    
    # ADJUST THIS: To hit 5GHz with 200nm x 200nm, 
    # Jc needs to be roughly 6.5e5 A/m^2
    Jc = 650000 
    
    # Extraction helper
    def get_m(key):
        val = str(qubit_obj.options[key]).lower()
        num = float(val.replace('um','').replace('nm','').replace('mm','').strip())
        if 'nm' in val: return num * 1e-9
        return num * 1e-6

    # 1. Geometry from Qubit
    w_m = get_m('jj_width')
    h_m = get_m('jj_height')
    pw_m = get_m('pad_width')
    ph_m = get_m('pad_height')
    pg_m = get_m('pad_gap')

    # 2. Inductance & Ej (Original Script Logic)
    area_m2 = w_m * h_m
    ic_a = Jc * area_m2
    lj_h = phi0 / (2 * np.pi * ic_a)
    lj_nh = lj_h * 1e9
    
    ej_joules = (ic_a * phi0) / (2 * np.pi)
    ej_ghz = ej_joules / h / 1e9

    # 3. Capacitance & Ec (Original Script Logic)
    eps_eff = (1 + 11.7) / 2.0
    # Csum approximation for transmon pads
    c_sum_f = 1.3 * eps0 * eps_eff * (pw_m * ph_m) / pg_m
    ec_joules = (e**2) / (2 * c_sum_f)
    ec_ghz = ec_joules / h / 1e9

    # 4. Final Frequency
    # f01 = sqrt(8*Ej*Ec) - Ec
    f01_ghz = np.sqrt(8 * ej_ghz * ec_ghz) - ec_ghz
    
    return {
        "Lj_nH": lj_nh,
        "Cq_fF": c_sum_f * 1e15,
        "Ej_GHz": ej_ghz,
        "Ec_MHz": ec_ghz * 1000,
        "f01_GHz": f01_ghz,
        "Ej_Ec": ej_ghz / ec_ghz
    }

# Run Verification
res = verify_qubit_physics(q1)
Ej_ghz = res['Ej_GHz']

print(f"--- Corrected Results ---")
print(f"L_j: {res['Lj_nH']:.2f} nH (Target: ~13nH)")
print(f"C_q: {res['Cq_fF']:.2f} fF (Target: ~70-90fF)")
print(f"f01: {res['f01_GHz']:.2f} GHz (Target: 5.0GHz)")
print(f"Ratio Ej/Ec: {res['Ej_Ec']:.1f}")

--- Corrected Results ---
L_j: 12.66 nH (Target: ~13nH)
C_q: 93.19 fF (Target: ~70-90fF)
f01: 4.43 GHz (Target: 5.0GHz)
Ratio Ej/Ec: 62.1


In [6]:
from qiskit_metal.analyses.quantization import EPRanalysis

In [7]:
eig_qb = EPRanalysis(design, "hfss")

In [8]:
# example: update single setting
eig_qb.sim.setup.max_passes = 6
eig_qb.sim.setup.vars.Lj =  Ej_ghz
# example: update multiple settings
eig_qb.sim.setup_update(max_delta_f = 0.4, min_freq_ghz = 1.1)

eig_qb.sim.setup

{'name': 'Setup',
 'reuse_selected_design': True,
 'reuse_setup': True,
 'min_freq_ghz': 1.1,
 'n_modes': 1,
 'max_delta_f': 0.4,
 'max_passes': 6,
 'min_passes': 1,
 'min_converged': 1,
 'pct_refinement': 30,
 'basis_order': 1,
 'vars': {'Lj': 12.913907996453837, 'Cj': '0 fF'}}

In [9]:
# Connect to the renderer and specify the Student 2025 R2 version string
ansys_renderer = eig_qb.sim.renderer
ansys_renderer.options['hfss_v3d_executable'] = 'AnsoftHfss.HfssApp.25.2'
ansys_renderer.options['get_simulation_terminal'] = False

In [13]:
eig_qb.sim.run(name="Qbit", components=['q1'], open_terminations=[], box_plus_buffer = False)
eig_qb.sim.plot_convergences()

com_error: (-2147221005, 'Invalid class string', None, None)

In [2]:
import win32com.client
app = win32com.client.Dispatch("Ansoft.ElectronicsDesktop")

com_error: (-2147221005, 'Invalid class string', None, None)

In [3]:
import win32com.client
app = win32com.client.Dispatch("Ansoft.ElectronicsDesktopStudent.2025.2")

com_error: (-2147221005, 'Invalid class string', None, None)

In [7]:
import win32com.client
print(win32com.__file__)

C:\Users\madhu\miniconda3\envs\quantum\lib\site-packages\win32com\__init__.py
